# Source Taxi Zone Lookup

This notebook performs a **one-time static load** of the NYC TLC Taxi Zone lookup table from the [NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page) public dataset into a Unity Catalog Delta table.

## What This Notebook Does

1. **Accepts parameters** for the target environment and an optional sink reset.
2. **Optionally resets** the Delta sink table and storage path to allow a clean re-load.
3. **Creates an external Unity Catalog Volume** backed by an Azure Data Lake Storage Gen2 inbound container.
4. **Downloads** the `taxi_zone_lookup.csv` file from the TLC CloudFront distribution directly into the Unity Catalog Volume.
5. **Reads, casts, and enriches** the CSV with explicit column types and audit metadata columns (execution ID, timestamp, source file name).
6. **Writes** the data as a Delta table in the ODS layer using `overwrite` mode.

## Source File
`https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv`

## Target Table
`raw_edl_dev_poc.{ENVIRONMENT}.taxi_zone_lookup`  
Stored in the ODS Azure Data Lake Storage container. This is a small reference table (~265 rows) and is not partitioned.

## Schema
| Column | Type | Description |
|---|---|---|
| `LocationID` | `int` | Unique identifier for the taxi zone. Referenced by `PULocationID` and `DOLocationID` in the trip data. |
| `Borough` | `string` | NYC borough the zone belongs to (e.g. Manhattan, Brooklyn). |
| `Zone` | `string` | Name of the taxi zone. |
| `service_zone` | `string` | Service category (e.g. Yellow Zone, Boro Zone, Airports). |

## Parameters

The following widgets control notebook execution. They can be set manually when running interactively, or passed programmatically via a Databricks job or workflow.

| Parameter | Default | Description |
|---|---|---|
| `ENVIRONMENT` | _(required)_ | Target environment name. Used to scope the Unity Catalog schema and ADLS paths. For this demo it is your name all lowercase. |
| `RESET_SINK` | `False` | When `True`, drops the existing Delta table and removes all ODS data for a clean re-load. **Use with caution in production.** |

In [ ]:
# Create Notebook Parameter Widgets
dbutils.widgets.removeAll()
dbutils.widgets.text("ENVIRONMENT", "", "Environment")
dbutils.widgets.dropdown("RESET_SINK", "False", ["True", "False"], "Reset Sink Location?")

In [ ]:
ENVIRONMENT = dbutils.widgets.get("ENVIRONMENT")
RESET_SINK = (dbutils.widgets.get("RESET_SINK") == "True")

## Imports

In [ ]:
import os
import requests
import uuid

from pyspark.sql.functions import *
from pyspark.sql.types import IntegerType, StringType, StructField, StructType

## Configuration

Defines source and destination paths based on the supplied parameters.

- **`SOURCE`** — Full URL to the static taxi zone lookup CSV on the TLC CloudFront distribution.
- **`INBOUND_PATH`** — ADLS Gen2 path for the raw inbound landing zone (used as the Unity Catalog Volume backing location).
- **`VOLUME_URL`** — Unity Catalog Volume path used to write the downloaded file from the driver node.
- **`ODS_PATH`** — ADLS Gen2 path for the Operational Data Store layer where the Delta table is written.
- **`TARGET_TABLE`** — Fully qualified Unity Catalog table name: `raw_edl_dev_poc.{ENVIRONMENT}.taxi_zone_lookup`.

In [ ]:
SOURCE         = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
DEST_PATH      = "abfss://{CONTAINER}@wbcdadpseudodlsa.dfs.core.windows.net/{ENVIRONMENT}"

CATALOG_NAME   = 'raw_edl_dev_poc'
SCHEMA_NAME    = ENVIRONMENT

INBOUND_PATH   = DEST_PATH.format(CONTAINER='inbound', ENVIRONMENT=ENVIRONMENT)
VOLUME_URL     = f'/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/inbound'
ODS_PATH       = DEST_PATH.format(CONTAINER='ods', ENVIRONMENT=ENVIRONMENT)

TARGET_FILE_NAME = "taxi_zone_lookup.csv"
TARGET           = f"{VOLUME_URL}/{TARGET_FILE_NAME}"

TARGET_LOCATION  = ODS_PATH + "/taxi_zone_lookup"
TARGET_TABLE     = f"{CATALOG_NAME}.{SCHEMA_NAME}.taxi_zone_lookup"

## Reset Sink (Optional)

When `RESET_SINK` is `True`, drops the Delta table from Unity Catalog and recursively deletes all files at the ODS target location. This allows a full re-load of the table from scratch.

> **Warning:** This is a destructive operation. Only enable this in non-production environments or when a clean reload is explicitly required.

In [ ]:
if RESET_SINK:
    spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE}")
    dbutils.fs.rm(TARGET_LOCATION, True)

## Create Inbound Volume

Ensures the ADLS Gen2 inbound directory exists and registers it as an external Unity Catalog Volume under `{catalog}.{schema}.inbound`. This volume is used as the landing zone for the downloaded CSV file and is idempotent — it will not overwrite an existing volume.

In [ ]:
# Create Inbound Location
dbutils.fs.mkdirs(INBOUND_PATH)

# Create Volume
spark.sql(f"""
CREATE EXTERNAL VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.inbound
  LOCATION '{INBOUND_PATH}'
  COMMENT 'Inbound Folder for {ENVIRONMENT}';
""")

## Download Source File

Downloads the static `taxi_zone_lookup.csv` file from the TLC CloudFront endpoint directly into the Unity Catalog Volume. This is a small file (~10 KB) so it is downloaded in a single request rather than chunked.

In [ ]:
print(f"Downloading from: {SOURCE}")

response = requests.get(SOURCE, timeout=30)
response.raise_for_status()

with open(TARGET, "wb") as f:
    f.write(response.content)

size_kb = os.path.getsize(TARGET) / 1024
print(f"Saved to {TARGET} ({size_kb:.1f} KB)")

## Transform and Write to Delta Table

Reads the downloaded CSV into a Spark DataFrame using an explicit schema to avoid type inference ambiguity, and appends the following audit/lineage columns:

| Column | Type | Description |
|---|---|---|
| `source_execution_id` | `string` | UUID generated at runtime to identify this specific notebook run. |
| `source_execution_dt` | `timestamp` | Timestamp of when this notebook run executed. |
| `source_file` | `string` | Name of the source CSV file that was downloaded. |

The DataFrame is written to the Delta table using `overwrite` mode, replacing the full table on each run. Since this is a static reference dataset with no partitioning, a full overwrite is the correct and simplest strategy.

In [ ]:
schema = StructType([
    StructField("LocationID",   IntegerType(), nullable=False),
    StructField("Borough",      StringType(),  nullable=True),
    StructField("Zone",         StringType(),  nullable=True),
    StructField("service_zone", StringType(),  nullable=True),
])

df = (spark.read
    .format("csv")
    .option("header", "true")
    .schema(schema)
    .load(TARGET))

df = df.withColumns({
    "source_execution_id": lit(str(uuid.uuid4())),
    "source_execution_dt": current_timestamp(),
    "source_file":         lit(TARGET_FILE_NAME),
})

(df.write
    .format("delta")
    .option("path", TARGET_LOCATION)
    .option("mergeSchema", "true")
    .mode("overwrite")
    .saveAsTable(TARGET_TABLE))

print(f"Written {df.count()} rows to {TARGET_TABLE}")